# Z-stack + Time-points automation — SPAD/SP5
Ludovic Leconte — Institut Pasteur de Montevideo

**Architecture**
- **pymmcore-plus** — contrôle de la platine Leica CTR6500 (FocusDrive) via Micro-Manager 2.0 (COM4)
- **BrightEyes REST API** — déclenche les acquisitions FLIM sur `http://127.0.0.1:8000`
- **brighteyes-ism** — post-processing optionnel (APR + FRC + diagnostic) en fin de script

**Pré-requis avant lancement**
1. Fermer Micro-Manager (sinon conflit COM4)
2. Lancer BrightEyes-MCS et démarrer manuellement le serveur HTTP depuis le ScriptLauncher : `FastAPIServerThread(main_window)`
3. Configurer dans BrightEyes : détecteur (SPAD/PMT), pixels XY, range, dwell time, time bins, dossier de sauvegarde
4. Faire la mise au point manuellement sur le contrôleur Leica — NE PAS toucher la mise au point après avoir lancé ce script

In [ ]:
import os
import time
import requests
import datetime
import json
from pathlib import Path
import tkinter as tk
from tkinter import filedialog, messagebox

import pymmcore_plus

print('✓ Imports OK')

## 1. Paramètres d'acquisition + sélection du dossier de sauvegarde

In [ ]:
# ============ PARAMETRES UTILISATEUR ============
Z_RANGE_UM   = 5.0     # épaisseur totale du stack en µm (sera centré sur z_origin)
Z_STEP_UM    = 0.3     # pas en µm (Nyquist axial typique ~0.3 µm à NA=1.4)
N_TIMEPOINTS = 1       # nombre de time-points (1 = un seul stack)
DELTA_T_S    = 60.0    # intervalle entre deux time-points en secondes

# Backlash compensation (FocusDrive Leica)
BACKLASH_OVERSHOOT_UM = 20.0   # descente compensatrice avant retour à z_origin

# Timeouts
STAGE_SETTLE_S        = 0.2    # stabilisation mécanique après mouvement Z
ACQ_START_TIMEOUT_S   = 5.0    # attente que BrightEyes passe en 'acquisition'
ACQ_FINISH_TIMEOUT_S  = 120.0  # attente max pour qu'un plan finisse

# Micro-Manager configuration
MM_CONFIG = r'C:\Users\prism\Documents\micromanager\MMConfig_leica.cfg'   # ou MMConfig_demo.cfg
MM_PATH   = r'C:\Program Files\Micro-Manager-2.0'

# BrightEyes REST API
BRIGHTEYES_URL = 'http://127.0.0.1:8000'

# ============ SELECTION DU DOSSIER ============
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
PARENT_FOLDER = filedialog.askdirectory(title='Choisir le dossier parent où créer le stack')
root.destroy()

if not PARENT_FOLDER:
    raise RuntimeError('Aucun dossier sélectionné.')

# Création d'un sous-dossier daté zstack_YYYYMMDD_HHMM
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = os.path.join(PARENT_FOLDER, f'zstack_{timestamp}')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Dossier de sauvegarde : {SAVE_DIR}')
print(f'Z-stack : {Z_RANGE_UM} µm, pas {Z_STEP_UM} µm → {int(Z_RANGE_UM/Z_STEP_UM)+1} plans')
print(f'Time-points : {N_TIMEPOINTS} (interval {DELTA_T_S}s)')

## 2. Connexion à Micro-Manager (FocusDrive Leica CTR6500)

In [ ]:
# Charger la configuration Micro-Manager
mmc = pymmcore_plus.CMMCorePlus()
mmc.setDeviceAdapterSearchPaths([MM_PATH])
mmc.loadSystemConfiguration(MM_CONFIG)

z_device = mmc.getFocusDevice()
xy_device = mmc.getXYStageDevice()
print(f'Focus device : {z_device}')
print(f'XY stage     : {xy_device}')

# IMPORTANT : le CTR6500 ne rapporte sa vraie position absolue qu'après un mouvement.
# Au démarrage, getPosition() peut retourner 0 ou une valeur périmée.
# On force un petit mouvement + compensation backlash pour obtenir la vraie position.
print('\nInitialisation du CTR6500 (mouvement + compensation backlash)...')
pos_raw = float(mmc.getPosition(z_device))
print(f'  Position brute (potentiellement fausse) : {pos_raw:.3f} µm')

# Mouvement relatif de +0.5 µm pour forcer le contrôleur à reporter sa vraie position
mmc.setRelativePosition(z_device, 0.5)
mmc.waitForDevice(z_device)
time.sleep(0.5)
pos_up = float(mmc.getPosition(z_device))
print(f'  Après +0.5 µm            : {pos_up:.3f} µm')

# Retour à la position de départ avec compensation backlash (descente 20 µm + remontée)
target = pos_up - 0.5
mmc.setPosition(z_device, target - BACKLASH_OVERSHOOT_UM)
mmc.waitForDevice(z_device)
time.sleep(0.5)
mmc.setPosition(z_device, target)
mmc.waitForDevice(z_device)
time.sleep(0.5)

z_origin = float(mmc.getPosition(z_device))
print(f'  Après compensation backlash : {z_origin:.3f} µm')
print(f'\nz_origin     : {z_origin:.3f} µm  (point focal vérifié)')
print(f'Dérive d’init  : {z_origin - pos_up + 0.5:+.3f} µm (devrait être proche de 0)')

## 3. Fonctions utilitaires

In [ ]:
def move_z(target_um, settle_s=STAGE_SETTLE_S):
    """Move FocusDrive to absolute Z position and wait for stabilization."""
    mmc.setPosition(z_device, target_um)
    mmc.waitForDevice(z_device)
    time.sleep(settle_s)
    return float(mmc.getPosition(z_device))

def return_to_origin_no_backlash(origin_um, overshoot_um=BACKLASH_OVERSHOOT_UM):
    """Return to z_origin with backlash compensation.
    The Leica FocusDrive has up to 5 µm of backlash when reversing direction.
    We always approach origin from BELOW (descend first, then ascend)."""
    move_z(origin_um - overshoot_um)
    actual = move_z(origin_um)
    delta = actual - origin_um
    print(f'  Return to origin : actual={actual:.3f} µm, delta={delta:+.3f} µm')
    return actual

def be_set(payload):
    """BrightEyes REST API : PUT /set with a JSON payload."""
    r = requests.put(f'{BRIGHTEYES_URL}/set', json=payload, timeout=5)
    r.raise_for_status()
    return r.json()

def be_get(endpoint):
    """BrightEyes REST API : GET on an endpoint."""
    r = requests.get(f'{BRIGHTEYES_URL}{endpoint}', timeout=5)
    r.raise_for_status()
    return r.json()

def be_full_state():
    """Return the full BrightEyes state as a dict (handles double-encoded JSON)."""
    raw = be_get('/full_state')
    if isinstance(raw, str):
        return json.loads(raw)
    return raw

def be_trigger_and_wait(target_folder, desired_name, timeout_s=ACQ_FINISH_TIMEOUT_S):
    """Trigger one acquisition on BrightEyes and wait for completion.
    
    BrightEyes chooses its own filename (no /set filename available).
    We watch completed_acquisition_count to detect the end, then we rename
    last_saved_filename to our naming convention.
    
    Args:
        target_folder : output folder (passed via defaultFolder)
        desired_name  : name we want for the saved file (e.g. 'T001_Z012_zXXX.h5')
    Returns:
        (acq_seconds, final_path) of the renamed file
    """
    # 1. Set output folder
    be_set({'defaultFolder': target_folder.replace('\\', '/')})
    
    # 2. Snapshot counter BEFORE
    state_before = be_full_state()
    count_before = state_before['completed_acquisition_count']
    
    # 3. Trigger acquisition
    t0 = time.time()
    be_get('/cmd/acquisition')
    
    # 4. Poll completed_acquisition_count until it increments
    while time.time() - t0 < timeout_s:
        state = be_full_state()
        if state['completed_acquisition_count'] > count_before:
            elapsed = time.time() - t0
            saved = state.get('last_saved_filename')
            if not saved or not os.path.exists(saved):
                raise RuntimeError(f'Acquisition done but last_saved_filename invalid: {saved}')
            # 5. Rename to our convention (with retry: BrightEyes may still hold the file)
            final_path = os.path.join(target_folder, desired_name)
            renamed = False
            for attempt in range(20):   # up to 10s of retries
                try:
                    os.replace(saved, final_path)
                    renamed = True
                    break
                except PermissionError:
                    time.sleep(0.5)
                except Exception as e:
                    print(f'  ⚠ Rename error ({e}), keeping {saved}')
                    final_path = saved
                    break
            if not renamed and final_path != saved:
                print(f'  ⚠ Rename failed after retries, keeping {saved}')
                final_path = saved
            return elapsed, final_path
        time.sleep(0.1)
    raise RuntimeError(f'Acquisition did not complete within {timeout_s}s')

print('✓ Fonctions utilitaires OK')

## 4. Sauvegarde du log avec la config BrightEyes en cours

In [ ]:
# Récupérer la config courante de BrightEyes
try:
    be_gui_config = be_get('/gui/')
    print('✓ Configuration BrightEyes récupérée via /gui/')
except Exception as e:
    be_gui_config = {'error': str(e)}
    print(f'⚠ Impossible de lire /gui/ : {e}')

# Calcul des positions Z — UPWARD ONLY (workaround : pymmcore ne descend pas)
# z_origin est la position de départ (focus actuel) = bas du stack
# On monte progressivement jusqu'à z_origin + Z_RANGE_UM
z_min = z_origin
z_max = z_origin + Z_RANGE_UM
z_positions = []
z = z_min
while z <= z_max + 1e-6:
    z_positions.append(round(z, 4))
    z += Z_STEP_UM
n_planes = len(z_positions)

print('⚠ Mode UPWARD-ONLY : le stack démarre au focus actuel et MONTE.')
print(f'  Mets ta mise au point au PLAN LE PLUS BAS de la zone d’intérêt.')

# Ecrire le log
log_path = os.path.join(SAVE_DIR, 'acquisition_log.txt')
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('Z-stack + Time-points acquisition log\n')
    f.write('=' * 60 + '\n')
    f.write(f'Date           : {datetime.datetime.now().isoformat()}\n')
    f.write(f'Save folder    : {SAVE_DIR}\n\n')
    f.write(f'Z origin       : {z_origin:.3f} µm\n')
    f.write(f'Z range        : {Z_RANGE_UM} µm  (from {z_min:.3f} to {z_max:.3f})\n')
    f.write(f'Z step         : {Z_STEP_UM} µm\n')
    f.write(f'N planes       : {n_planes}\n')
    f.write(f'N timepoints   : {N_TIMEPOINTS}\n')
    f.write(f'Delta T        : {DELTA_T_S} s\n')
    f.write(f'Backlash comp  : {BACKLASH_OVERSHOOT_UM} µm overshoot\n\n')
    f.write('Micro-Manager config:\n')
    f.write(f'  MM_CONFIG    : {MM_CONFIG}\n')
    f.write(f'  Z device     : {z_device}\n\n')
    f.write('BrightEyes /gui/ snapshot:\n')
    f.write(json.dumps(be_gui_config, indent=2, ensure_ascii=False))
    f.write('\n\n' + '=' * 60 + '\n')
    f.write('Per-plane log:\n')

print(f'Log : {log_path}')
print(f'Plans Z : {n_planes} ({z_min:.3f} → {z_max:.3f} µm)')

## 5. Boucle d'acquisition

In [ ]:
t_start_global = time.time()

with open(log_path, 'a', encoding='utf-8') as logf:
    for t_idx in range(N_TIMEPOINTS):
        t_start_tp = time.time()
        print(f'\n{"="*60}')
        print(f'TIME-POINT {t_idx+1}/{N_TIMEPOINTS}')
        print(f'{"="*60}')
        logf.write(f'\nT{t_idx+1}/{N_TIMEPOINTS} — start = {datetime.datetime.now().isoformat()}\n')
        
        # MODE UPWARD-ONLY : on part de z_origin (focus actuel) et on monte
        # Pas de mouvement initial vers le bas (pymmcore ne descend pas sur ton setup)
        move_z(z_min)
        
        # Acquire each plane
        for z_idx, z_target in enumerate(z_positions):
            t_start_plane = time.time()
            
            # Move stage (always approaching from below = no backlash)
            actual_z = move_z(z_target)
            
            # Nom de fichier souhaité : T001_Z012_zX.XXXum.h5
            fname = f'T{t_idx+1:03d}_Z{z_idx+1:03d}_z{actual_z:.3f}um.h5'
            
            # Trigger BrightEyes acquisition + renommage automatique
            try:
                acq_time, final_path = be_trigger_and_wait(SAVE_DIR, fname)
                plane_time = time.time() - t_start_plane
                msg = f'  T{t_idx+1:03d} Z{z_idx+1:03d}/{n_planes} z={actual_z:7.3f} µm  acq={acq_time:5.2f}s  total={plane_time:5.2f}s  → {os.path.basename(final_path)}'
                print(msg)
                logf.write(msg + '\n')
                logf.flush()
            except Exception as e:
                err = f'  ✗ ECHEC plan Z{z_idx+1} (z={actual_z:.3f} µm) : {e}'
                print(err)
                logf.write(err + '\n')
                logf.flush()
                # On continue le stack
        
        # MODE UPWARD-ONLY : on ne peut PAS revenir à z_origin (descente impossible)
        # Le focus est laissé au sommet du stack — à régler manuellement entre time-points si besoin
        current_z = float(mmc.getPosition(z_device))
        print(f'  Stack terminé à z = {current_z:.3f} µm (focus laissé en haut)')
        
        tp_duration = time.time() - t_start_tp
        logf.write(f'T{t_idx+1} duration: {tp_duration:.1f}s\n')
        
        # Wait until next time-point (if any)
        if t_idx < N_TIMEPOINTS - 1:
            wait_remaining = DELTA_T_S - tp_duration
            if wait_remaining > 0:
                print(f'  Wait {wait_remaining:.1f}s before next time-point...')
                time.sleep(wait_remaining)
            else:
                print(f'  ⚠ Time-point pris plus de temps que DELTA_T_S — pas d\'attente.')
    
    total_duration = time.time() - t_start_global
    logf.write(f'\n{"="*60}\n')
    logf.write(f'Total duration: {total_duration:.1f}s ({total_duration/60:.1f} min)\n')

print(f'\n{"="*60}')
print(f'ACQUISITION TERMINEE — durée totale : {total_duration:.1f}s ({total_duration/60:.1f} min)')
print(f'Log : {log_path}')
print(f'{"="*60}')

## 6. Post-processing optionnel (APR + FRC + diagnostic)

Une fenêtre va demander si tu veux lancer le post-processing sur les fichiers HDF5 du dossier.

In [ ]:
# Ask user if they want to run post-processing
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
do_post = messagebox.askyesno(
    'Post-processing',
    f'Acquisition terminée ({n_planes * N_TIMEPOINTS} fichiers).\n\n'
    f'Lancer le post-processing APR + FRC + diagnostic\n'
    f'sur tous les fichiers du dossier ?\n\n'
    f'{SAVE_DIR}'
)
root.destroy()

if not do_post:
    print('Post-processing skippé. Les fichiers HDF5 sont dans :')
    print(f'  {SAVE_DIR}')
else:
    print('Lancement du post-processing...')

In [ ]:
if do_post:
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    import glob
    import csv
    import traceback
    import tifffile as tiff
    
    from brighteyes_ism.dataio.mcs import load
    import brighteyes_ism.analysis.Graph_lib as gra
    import brighteyes_ism.analysis.APR_lib as apr
    import brighteyes_ism.analysis.FRC_lib as frc
    
    USF = 10
    REF_CHANNEL = 12
    DPI = 150
    
    post_dir = os.path.join(SAVE_DIR, 'diagnostic')
    os.makedirs(post_dir, exist_ok=True)
    
    # Lister les fichiers HDF5
    h5_files = sorted(glob.glob(os.path.join(SAVE_DIR, '*.h5')))
    h5_files = [f for f in h5_files if not any(
        s in os.path.basename(f).lower() for s in ['_calib', '_irf', '_ref']
    )]
    print(f'{len(h5_files)} fichier(s) HDF5 à traiter')
    
    def save_25ch_tiff(channels_yxc, filepath, pixel_size_um):
        stack = np.moveaxis(channels_yxc, -1, 0).astype(np.float32)
        resolution = (1e4 / pixel_size_um, 1e4 / pixel_size_um)
        tiff.imwrite(filepath, stack, resolution=resolution,
                     metadata={'unit': 'um', 'axes': 'CYX'}, imagej=True)
    
    def save_2d_tiff(img, filepath, pixel_size_um):
        resolution = (1e4 / pixel_size_um, 1e4 / pixel_size_um)
        tiff.imwrite(filepath, img.astype(np.float32),
                     resolution=resolution, metadata={'unit': 'um'}, imagej=True)
    
    def compute_shifts(data, usf=10, ref=12):
        dset_full = np.sum(data, axis=(0, 1, 4))
        sf, _ = apr.APR(dset_full, usf, ref)
        data_t = data.sum(axis=(0, 1))
        se, _ = apr.APR(data_t[:, :, ::2,  :].sum(axis=-2), usf, ref)
        so, _ = apr.APR(data_t[:, :, 1::2, :].sum(axis=-2), usf, ref)
        return np.asarray(sf), np.asarray(se), np.asarray(so)
    
    def draw_quiver(ax, shift_xy_nm, color, title, n_side=5, ref=12):
        grid = np.zeros((n_side, n_side, 2))
        for ch in range(n_side * n_side):
            row, col = ch // n_side, ch % n_side
            grid[row, col] = shift_xy_nm[ch]
        X, Y = np.meshgrid(np.arange(n_side), np.arange(n_side))
        max_mag = float(np.linalg.norm(shift_xy_nm, axis=1).max())
        scale = max(max_mag / 0.4, 1.0)
        ax.quiver(X, Y, grid[:, :, 0], -grid[:, :, 1],
                  scale=scale, scale_units='xy', angles='xy',
                  color=color, width=0.008, headwidth=4, headlength=5, zorder=2)
        for ch in range(n_side * n_side):
            row, col = ch // n_side, ch % n_side
            ax.text(col, row, f'{ch}', ha='center', va='center',
                    fontsize=7, color='gray', zorder=1)
        ref_row, ref_col = ref // n_side, ref % n_side
        ax.scatter([ref_col], [ref_row], color='orange', s=100,
                   edgecolors='black', zorder=5, label=f'ref ch{ref}')
        ax.set_xlim(-0.5, n_side - 0.5)
        ax.set_ylim(-0.5, n_side - 0.5)
        ax.invert_yaxis()
        ax.set_xticks(range(n_side)); ax.set_yticks(range(n_side))
        ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
        ax.set_title(title, fontsize=10)
        ax.legend(loc='upper right', fontsize=8)
    
    results = []
    failed = []
    
    for i, filepath in enumerate(h5_files):
        basename = os.path.splitext(os.path.basename(filepath))[0]
        print(f'\n[{i+1}/{len(h5_files)}] {basename}')
        try:
            data, meta = load(filepath)
            pixel_size_um = meta.dx
            pixel_size_nm = meta.dx * 1000
            dwell_us = meta.pxdwelltime
            
            # Project + APR
            dset = np.sum(data, axis=(0, 1, 4))
            ny, nx, n_ch = dset.shape
            n_side = int(np.sqrt(n_ch))
            shift_vec, apr_rec = apr.APR(dset, USF, REF_CHANNEL)
            
            img_open  = dset.sum(-1)
            img_close = dset[:, :, REF_CHANNEL]
            img_ism   = apr_rec.sum(-1)
            
            # Images panel
            fig, ax = plt.subplots(1, 3, figsize=(18, 5))
            gra.ShowImg(img_open,  pixel_size_um, dwell_us, fig=fig, ax=ax[0])
            gra.ShowImg(img_close, pixel_size_um, dwell_us, fig=fig, ax=ax[1])
            gra.ShowImg(img_ism,   pixel_size_um, dwell_us, fig=fig, ax=ax[2])
            ax[0].set_title('Confocal — open pinhole')
            ax[1].set_title(f'Confocal — close pinhole (ch{REF_CHANNEL})')
            ax[2].set_title('ISM + APR')
            fig.suptitle(basename, fontsize=10)
            plt.tight_layout()
            fig.savefig(os.path.join(post_dir, f'{basename}_images.png'), dpi=DPI, bbox_inches='tight')
            plt.close(fig)
            
            # Shifts + fingerprint + quivers
            sf, se, so = compute_shifts(data, usf=USF, ref=REF_CHANNEL)
            shift_full_nm = sf * pixel_size_nm
            shift_even_nm = se * pixel_size_nm
            shift_odd_nm  = so * pixel_size_nm
            fingerprint     = dset.sum(axis=(0, 1)).reshape(n_side, n_side)
            fingerprint_pct = 100.0 * fingerprint / fingerprint.max()
            
            fig = plt.figure(figsize=(15, 11))
            fig.suptitle(f'{basename} — shifts diagnostic', fontsize=12)
            ax1 = plt.subplot2grid((2, 6), (0, 0), colspan=3)
            ax2 = plt.subplot2grid((2, 6), (0, 3), colspan=3)
            ax_qf = plt.subplot2grid((2, 6), (1, 0), colspan=2)
            ax_qe = plt.subplot2grid((2, 6), (1, 2), colspan=2)
            ax_qo = plt.subplot2grid((2, 6), (1, 4), colspan=2)
            
            mags = np.linalg.norm(shift_full_nm, axis=1)
            sc = ax1.scatter(shift_full_nm[:, 0], shift_full_nm[:, 1], c=mags,
                             cmap='viridis', s=130, edgecolors='black', zorder=3)
            for k, (sx, sy) in enumerate(shift_full_nm):
                ax1.annotate(str(k), (sx, sy), fontsize=8, xytext=(5, 5), textcoords='offset points')
            ax1.axhline(0, color='gray', lw=0.5, ls='--')
            ax1.axvline(0, color='gray', lw=0.5, ls='--')
            ax1.set_xlabel('Shift$_x$ (nm)'); ax1.set_ylabel('Shift$_y$ (nm)')
            ax1.set_title('Shift vectors (full dataset)')
            ax1.set_aspect('equal'); ax1.grid(alpha=0.3)
            plt.colorbar(sc, ax=ax1, label='|shift| (nm)', fraction=0.046, pad=0.04)
            
            im = ax2.imshow(fingerprint_pct, cmap='hot', vmin=0, vmax=100)
            ax2.set_title('Fingerprint (% of max signal)')
            ax2.set_xticks(range(n_side)); ax2.set_yticks(range(n_side))
            for r in range(n_side):
                for c in range(n_side):
                    ch_idx = r * n_side + c
                    pct = fingerprint_pct[r, c]
                    color = 'black' if pct > 55 else 'white'
                    ax2.text(c, r - 0.18, f'{ch_idx}', ha='center', va='center',
                             color=color, fontsize=9, fontweight='bold')
                    ax2.text(c, r + 0.22, f'{pct:.1f}%', ha='center', va='center',
                             color=color, fontsize=10)
            center = n_side // 2
            ax2.add_patch(Rectangle((center - 0.5, center - 0.5), 1, 1,
                                    fill=False, edgecolor='cyan', lw=2))
            plt.colorbar(im, ax=ax2, label='Normalized signal (%)', fraction=0.046, pad=0.04)
            
            draw_quiver(ax_qf, shift_full_nm, 'steelblue',
                        f'Full (max={mags.max():.0f} nm)', n_side, REF_CHANNEL)
            draw_quiver(ax_qe, shift_even_nm, 'coral',
                        f'Even (max={np.linalg.norm(shift_even_nm,axis=1).max():.0f} nm)', n_side, REF_CHANNEL)
            draw_quiver(ax_qo, shift_odd_nm,  'seagreen',
                        f'Odd  (max={np.linalg.norm(shift_odd_nm, axis=1).max():.0f} nm)', n_side, REF_CHANNEL)
            plt.tight_layout()
            fig.savefig(os.path.join(post_dir, f'{basename}_shifts.png'), dpi=DPI, bbox_inches='tight')
            plt.close(fig)
            
            # FRC
            data_t = data.sum(axis=(0, 1))
            data_split = [data_t[:, :, ::2, :], data_t[:, :, 1::2, :]]
            images_sum, images_ism = [], []
            for n in range(2):
                img = data_split[n].sum(axis=-2)
                images_sum.append(img.sum(axis=-1))
                images_ism.append(apr.APR(img, USF, REF_CHANNEL)[1].sum(axis=-1))
            frc_sum = frc.FRC_resolution(images_sum[0], images_sum[1],
                                         px=pixel_size_um, method='fixed', smoothing='fit')
            frc_ism = frc.FRC_resolution(images_ism[0], images_ism[1],
                                         px=pixel_size_um, method='fixed', smoothing='fit')
            res_conf_nm = frc_sum[0] * 1000
            res_apr_nm  = frc_ism[0] * 1000
            gain = (res_conf_nm / res_apr_nm) if res_apr_nm else float('nan')
            print(f'   Confocal={res_conf_nm:.1f} nm, ISM+APR={res_apr_nm:.1f} nm, gain={gain:.2f}x')
            
            # TIFF exports
            save_25ch_tiff(dset,     os.path.join(post_dir, f'{basename}_channels_raw_25ch.tif'), pixel_size_um)
            save_25ch_tiff(apr_rec,  os.path.join(post_dir, f'{basename}_channels_apr_25ch.tif'), pixel_size_um)
            save_2d_tiff(img_open,  os.path.join(post_dir, f'{basename}_confocal_open.tif'),  pixel_size_um)
            save_2d_tiff(img_close, os.path.join(post_dir, f'{basename}_confocal_close.tif'), pixel_size_um)
            save_2d_tiff(img_ism,   os.path.join(post_dir, f'{basename}_ism_apr.tif'),        pixel_size_um)
            
            results.append({'file': basename, 'pixel_size_nm': pixel_size_nm,
                            'res_confocal_nm': res_conf_nm, 'res_apr_nm': res_apr_nm,
                            'gain': gain})
        except Exception as e:
            print(f'   ✗ {e}')
            traceback.print_exc()
            failed.append((basename, str(e)))
    
    # CSV récapitulatif
    if results:
        csv_path = os.path.join(post_dir, 'STACK_summary.csv')
        with open(csv_path, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['file', 'pixel_size_nm', 'res_confocal_nm', 'res_apr_nm', 'gain'])
            for r in results:
                w.writerow([r['file'], r['pixel_size_nm'],
                            f"{r['res_confocal_nm']:.1f}",
                            f"{r['res_apr_nm']:.1f}",
                            f"{r['gain']:.3f}"])
        print(f'\nCSV : {csv_path}')
    print(f'\nPost-processing terminé : {len(results)} OK, {len(failed)} échec(s)')